In [1]:
import os
import numpy as np
from netCDF4 import Dataset

# -----------------------------
# ANUGA .tms writer (based on yours)
# -----------------------------
def create_tms_manually(file_path, times, discharges):
    """Creates a NetCDF .tms file with cleaned metadata."""
    out_dir = os.path.dirname(file_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)

    with Dataset(file_path, "w", format="NETCDF4") as rootgrp:
        rootgrp.createDimension("time", len(times))

        t_var = rootgrp.createVariable("time", "f8", ("time",))
        q_var = rootgrp.createVariable("discharge", "f8", ("time",))

        t_var[:] = times
        q_var[:] = discharges

        # Standard ANUGA NetCDF attributes
        rootgrp.description = "Manual conversion for ANUGA"
        rootgrp.starttime = 0.0
        rootgrp.xllcorner = 0.0
        rootgrp.yllcorner = 0.0
        rootgrp.zone = -1
        rootgrp.projection = "UTM"
        rootgrp.datum = "WGS84"


def csv_to_tms(csv_path, tms_path, time_scale=24 * 3600.0, delimiter=","):
    """
    Reads a 2-column CSV with no header: (x, y)
    Writes .tms where time = x * time_scale and discharge = y
    """
    data = np.loadtxt(csv_path, delimiter=delimiter)

    if data.ndim == 1:
        # If the file has only 1 row, loadtxt returns shape (2,)
        if data.shape[0] != 2:
            raise ValueError("Expected exactly 2 columns (x, y).")
        data = data.reshape(1, 2)

    if data.shape[1] < 2:
        raise ValueError("Expected at least 2 columns (x, y).")

    x = data[:, 0]
    y = data[:, 1]

    times = x * time_scale
    discharges = y

    create_tms_manually(tms_path, times, discharges)


In [2]:
import pandas as pd

for i in range(1, 4):
    csv_to_tms(f"BC_{i}.csv", f"BC_{i}.tms")
    print(f"Done! Saved as BC_{i}.csv")


Done! Saved as BC_1.csv
Done! Saved as BC_2.csv
Done! Saved as BC_3.csv
